# 03b · Irigat 2024: regula radar Sentinel-1 + pivoți geometrici (D26, D27)
Fără clasificator optic și fără gardă pe NDVI de vară (evitarea circularității cu `loss_rel` din HR-VPP TPROD):

- **Pivoți geometrici (`data/out/pivots_2024.gpkg`):** $C_{\text{PP}} \ge 0{,}75$, $C_{\text{MBC}} \ge 0{,}60$, $A \ge 5\text{ ha}$, pe toate grupele.
- **Regula Sentinel-1 pe culturile de primăvară (`tip_cultura_2024 == 'primavara'`, Varianta B aprobată în D27):** aceeași orbită relativă, $\Delta t = 12\text{ zile}$ (iunie–august 2024), excluderea tranziției regionale `o7_0615_0627`, ploaie ERA5-Land $< 2\text{ mm}$ în ultimele 5 zile și $< 5\text{ mm}$ pe toate cele 12 zile, eveniment de umezire la $\Delta\text{VV} \ge +1{,}5\text{ dB}$.
- **Limite metodologice (D27 pct. 2):** *(a)* ferestrele valide sunt aproape doar în august $\rightarrow$ `„irigat”` = umezire detectată în august 2024; irigarea din iunie–iulie nu se vede; *(b)* pivoții sunt geometrici și pot fi inactivi în 2024.

In [1]:
import os, sys, json
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import (
    geometric_pivots, s1_spring_irrigation_rule,
    read_validation_labels_csv, olofsson_evaluation, block_bootstrap_median_diff
)
import numpy as np, pandas as pd, geopandas as gpd

P = gpd.read_parquet(OUT / 'parcels.parquet').to_crs(CRS)
T = pd.read_parquet(OUT / 'parcel_target.parquet')[['parcel_id', 'is_arable', 'tip_cultura_2024', 'loss_rel']]
S1 = pd.read_parquet(INTERIM / 'q2_s1_vv_arable.parquet')
ERA = pd.read_parquet(INTERIM / 'q2_era5_daily_arable.parquet')
print('Parcele:', len(P), '| S1 arabil:', S1.shape, '| ERA5 arabil:', ERA.shape)

Parcele: 17466 | S1 arabil: (13237, 35) | ERA5 arabil: (13237, 110)


In [2]:
# 1. Pivoți geometrici (strat separat, pe toate grupele, fără condiție NDVI)
piv_df = geometric_pivots(P.geometry)
P_piv = P[['parcel_id', 'geometry']].join(piv_df).merge(T, on='parcel_id')
P_piv['grup_afisare'] = np.where(P_piv.is_arable, P_piv.tip_cultura_2024, 'non-arabil')
piv_only = gpd.GeoDataFrame(P_piv[P_piv.is_pivot].copy(), geometry='geometry', crs=CRS)
piv_only.to_file(OUT / 'pivots_2024.gpkg', layer='pivoti', driver='GPKG')
piv_only.drop(columns='geometry').to_parquet(OUT / 'pivots_2024.parquet')
print(f'Total pivoți geometrici (C_PP>=0.75, C_MBC>=0.60, A>=5ha): {len(piv_only)} parcele ({piv_only.area_ha.sum():.1f} ha)')
for grp, gdf in piv_only.groupby('grup_afisare'):
    print(f'  {grp:14s}: {len(gdf):3d} parcele ({gdf.area_ha.sum():7.1f} ha)')

Total pivoți geometrici (C_PP>=0.75, C_MBC>=0.60, A>=5ha): 121 parcele (1810.1 ha)
  grup_incert   :   3 parcele (   45.1 ha)
  non-arabil    :  19 parcele (  119.0 ha)
  primavara     :  32 parcele (  369.8 ha)
  toamna        :  67 parcele ( 1276.2 ha)


In [3]:
# 2. Clasificare oficială în Varianta B (fără tranziția regională o7_0615_0627, D27 pct. 1) și calculul Panoului 3 (D27 pct. 5)
D = P_piv.merge(S1, on='parcel_id', how='left').merge(ERA, on='parcel_id', how='left')
spring = D[D.is_arable & D.tip_cultura_2024.eq('primavara')].copy().reset_index(drop=True)
c_cent = spring.geometry.centroid
spring['block'] = (c_cent.x // 5000).astype(int).astype(str) + '_' + (c_cent.y // 5000).astype(int).astype(str)

tr_all, res_raw = s1_spring_irrigation_rule(spring, exclude_transitions=())
tr_b, res_b = s1_spring_irrigation_rule(spring, exclude_transitions=('o7_0615_0627',))
tr_b.to_csv(OUT / 's1_transitions_2024.csv', index=False)

print('=== VARIANTA B OFICIALĂ (fără tranziția regională o7_0615_0627) ===')
for c_name in ('irigat', 'neirigat', 'nedeterminat'):
    m = res_b.clasa_irigare.eq(c_name)
    n_p = int(m.sum())
    ha_p = float(spring.loc[m, 'area_ha'].sum())
    print(f'  {c_name:14s}: {n_p:5d} parcele ({100*n_p/len(spring):5.1f}%) | {ha_p:8.1f} ha ({100*ha_p/spring.area_ha.sum():5.1f}%)')

spring['clasa_irigare'] = res_b['clasa_irigare']
p3_boot = block_bootstrap_median_diff(spring, group_col='clasa_irigare', value_col='loss_rel', block_col='block', n_boot=5000, seed=2024)
print('\n=== PANOU 3 (clase RADAR Varianta B pe toate parcelele de primăvară cu țintă, fără nedeterminat) ===')
for k in ('irigat', 'neirigat'):
    st = p3_boot[k]
    print(f"  {k:10s} (N={st['n']:4d}): mediană loss_rel = {st['median']*100:+.2f}% | P25–P75 = {st['p25']*100:+.2f}% … {st['p75']*100:+.2f}%")
print(f"  Diferență mediane (irigat − neirigat): {p3_boot['diff_median_pos_minus_neg']*100:+.2f} pp "
      f"(IC 95% bootstrap pe {p3_boot['n_blocks']} blocuri de 5 km: {p3_boot['diff_ci95_block_bootstrap'][0]*100:+.2f} … {p3_boot['diff_ci95_block_bootstrap'][1]*100:+.2f} pp)")

out_df = pd.DataFrame({
    'parcel_id': spring['parcel_id'],
    'area_ha': spring['area_ha'],
    'is_pivot': res_b['is_pivot'],
    'n_dry_obs': res_b['n_dry_obs'],
    'n_events_s1': res_b['n_events_s1'],
    'max_dvv_dry': res_b['max_dvv_dry'],
    'clasa_irigare': res_b['clasa_irigare'],
    'clasa_irigare_fara_0627': res_b['clasa_irigare'],
    'clasa_irigare_brut_cu_0627': res_raw['clasa_irigare'],
    'irrigated': res_b['clasa_irigare'].eq('irigat'),
})
out_df.to_parquet(OUT / 'irrigation_2024.parquet')
gpd.GeoDataFrame(out_df.merge(spring[['parcel_id', 'geometry']], on='parcel_id'), geometry='geometry', crs=CRS).to_file(
    OUT / 'irrigation_2024.gpkg', layer='irigare_primavara_2024', driver='GPKG'
)

=== VARIANTA B OFICIALĂ (fără tranziția regională o7_0615_0627) ===
  irigat        :   627 parcele ( 10.7%) |   2943.7 ha ( 13.2%)
  neirigat      :  4267 parcele ( 73.1%) |  16838.1 ha ( 75.5%)
  nedeterminat  :   942 parcele ( 16.1%) |   2533.6 ha ( 11.4%)



=== PANOU 3 (clase RADAR Varianta B pe toate parcelele de primăvară cu țintă, fără nedeterminat) ===
  irigat     (N= 506): mediană loss_rel = +31.95% | P25–P75 = +6.34% … +49.73%
  neirigat   (N=3645): mediană loss_rel = +29.96% | P25–P75 = +1.08% … +52.15%
  Diferență mediane (irigat − neirigat): +1.99 pp (IC 95% bootstrap pe 81 blocuri de 5 km: -4.33 … +7.15 pp)


In [4]:
# 3. Evaluarea pe labels.csv (Olofsson et al. 2014, D27 pct. 4 & 6)
weights_parcels = {c: int(res_b.clasa_irigare.eq(c).sum()) for c in ('irigat', 'neirigat', 'nedeterminat')}
weights_ha = {c: round(float(spring.loc[res_b.clasa_irigare.eq(c), 'area_ha'].sum()), 2) for c in ('irigat', 'neirigat', 'nedeterminat')}

metrics = {
    'method': 's1_vv_rule_plus_geometric_pivots_varianta_b_fara_o7_0615_0627',
    'excluded_transitions': ['o7_0615_0627'],
    'limits': [
        '(a) Ferestrele S1 valide sunt aproape doar în august -> „irigat” = umezire detectată în august 2024; irigarea din iunie–iulie nu se vede.',
        '(b) Pivoții sunt identificați geometric (C_PP >= 0,75, C_MBC >= 0,60, A >= 5 ha); un pivot poate fi inactiv în 2024.',
    ],
    'n_spring_parcels': int(len(spring)),
    'ha_spring_parcels': round(float(spring.area_ha.sum()), 2),
    'varianta_b': {c: {'n': weights_parcels[c], 'ha': weights_ha[c]} for c in ('irigat', 'neirigat', 'nedeterminat')},
    'panou3_radar_varianta_b': {
        'irigat': {
            'n': p3_boot['irigat']['n'],
            'p25_pct': round(p3_boot['irigat']['p25'] * 100, 2),
            'median_pct': round(p3_boot['irigat']['median'] * 100, 2),
            'p75_pct': round(p3_boot['irigat']['p75'] * 100, 2),
        },
        'neirigat': {
            'n': p3_boot['neirigat']['n'],
            'p25_pct': round(p3_boot['neirigat']['p25'] * 100, 2),
            'median_pct': round(p3_boot['neirigat']['median'] * 100, 2),
            'p75_pct': round(p3_boot['neirigat']['p75'] * 100, 2),
        },
        'diff_median_pct': round(p3_boot['diff_median_pos_minus_neg'] * 100, 2),
        'diff_ci95_pct': [round(v * 100, 2) for v in p3_boot['diff_ci95_block_bootstrap']],
        'n_blocks': p3_boot['n_blocks'],
        'n_boot': p3_boot['n_boot'],
        'nedeterminat_excluded': {'n': weights_parcels['nedeterminat'], 'ha': weights_ha['nedeterminat']},
    },
    'panou3_gate_passed': False,
}

labels_path = OUT / 'labels.csv'
if not labels_path.exists() or read_validation_labels_csv(labels_path)['irigat'].notna().sum() == 0:
    alt_path = INTERIM / 'label_sheets' / 'labels.csv'
    if alt_path.exists() and read_validation_labels_csv(alt_path)['irigat'].notna().sum() > 0:
        labels_path = alt_path

lbl_df = read_validation_labels_csv(labels_path) if labels_path.exists() else pd.DataFrame(columns=['parcel_id', 'irigat'])
n_filled = int(lbl_df['irigat'].notna().sum())
if n_filled == 0:
    print('labels.csv are încă 0 etichete completate; aștept completarea de către Patrick înainte de evaluarea Olofsson și Panoul 3.')
else:
    hints = pd.read_parquet(INTERIM / 'irrigation_label_hints.parquet')
    idx_sheet = pd.read_csv(INTERIM / 'label_sheets' / 'index.csv')[['numar', 'parcel_id', 'foaie']]
    eval_df = lbl_df.merge(hints, on='parcel_id').merge(idx_sheet, on='parcel_id').merge(spring[['parcel_id', 'loss_rel', 'area_ha']], on='parcel_id')
    olof_parcels = olofsson_evaluation(eval_df['clasa_regula'], eval_df['irigat'], {'irigat': weights_parcels['irigat'], 'neirigat': weights_parcels['neirigat']})
    olof_ha = olofsson_evaluation(eval_df['clasa_regula'], eval_df['irigat'], {'irigat': weights_ha['irigat'], 'neirigat': weights_ha['neirigat']})
    ua_irig = olof_parcels['user_accuracy']['irigat']['est']
    metrics['validation_olofsson_by_parcels'] = olof_parcels
    metrics['validation_olofsson_by_ha'] = olof_ha
    metrics['panou3_gate_passed'] = bool(ua_irig >= 0.70)
    print(f'Etichete completate: {n_filled}/80 | Acuratețe clasa irigat (direct din 40+40): {ua_irig:.1%} (IC 95%: {[round(x*100,1) for x in olof_parcels["user_accuracy"]["irigat"]["ci95"]]})')
    print(f'Acuratețe clasa neirigat (direct din 40+40): {olof_parcels["user_accuracy"]["neirigat"]["est"]:.1%}')
    print(f'Olofsson (pe parcele, fără nedeterminat): OA={olof_parcels["overall_accuracy"]["est"]:.1%}, Recall irigat={olof_parcels["producer_accuracy"]["irigat"]["est"]:.1%}')
    print(f'Olofsson (pe hectare, fără nedeterminat): OA={olof_ha["overall_accuracy"]["est"]:.1%}, Recall irigat={olof_ha["producer_accuracy"]["irigat"]["est"]:.1%}')
    if ua_irig < 0.70:
        wrong = eval_df[eval_df.irigat.notna() & ((eval_df.clasa_regula.eq('irigat') & (eval_df.irigat == 0)) | (eval_df.clasa_regula.eq('neirigat') & (eval_df.irigat == 1)))]
        print('OPRIRE (acuratețea clasei irigat < 70%). Parcele greșite:')
        print(wrong[['numar', 'foaie', 'parcel_id', 'clasa_regula', 'irigat', 'is_pivot', 'n_events_s1_fara_0627', 'max_dvv_dry', 'loss_rel']].to_string(index=False))

(OUT / 'irrigation_metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')

labels.csv are încă 0 etichete completate; aștept completarea de către Patrick înainte de evaluarea Olofsson și Panoul 3.


1178